# Phase 3 — DEV experiments (slot-free)

Shared channel setup with the Blind-A submission notebook (`phase3_blindA_submission.ipynb`), but NO submission path. Use this to try retrieval/rerank levers on the dev **final-turn** proxy (one trailing turn per dev session = how Blind-A is scored). Run cells 1–5 once, then run any experiment cell. Add new experiments as new cells at the bottom.

**Trains nothing.** Each model has its own notebook — load the pretrained artifacts here if an experiment needs them: K2 (LGBM) from `phase2_rerank.ipynb` (`K2_MODEL_PATH`), the ColBERT checkpoint from `phase2_colbert_finetune.ipynb` (`COLBERT_OUT_DIR`).

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)
try:                                   # only needed when RESPONDER=='gemini'
    os.environ['GEMINI_API_KEY']=os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY'); print('Gemini key ok')
except Exception as e: print('no GEMINI_API_KEY secret (only needed for the responder):', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate peft google-genai
!pip -q uninstall -y torchao   # transformers wants torchao>0.16 but Colab ships 0.10 and its check RAISES; we don't use it
import sys; sys.path.insert(0,'.')

In [ ]:
# OOM hygiene — MUST run before any import that pulls in JAX/torch (RESTART to apply mid-session).
# Root cause of the GPU OOMs: JAX preallocates 75% of VRAM on init, leaving PyTorch ~24% -> OOM.
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'   # JAX: allocate on demand (THE fix)
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
print('OOM-hygiene env set — RESTART if torch/jax were already imported')

## 3. Config (shared with the submission nb; only channel knobs matter here)

In [ ]:
# ── mode ──
BLIND=False                # this is the slot-free DEV experiments nb — keep False (no submission path here)
RESPONDER='stub'           # 'stub' (default predicted_response, read nDCG axis only) | 'gemini' (full composite)
DEFAULT_RESPONSE='ok'      # stub responder value written to every row (nDCG-only online eval; nb82 convention)
SMOKE=0                    # >0 caps serve turns for a quick pipeline check (0 = all)
SEED=42

# ── splits ──
ORG='talkpl-ai'
DEV_SESSIONS=1000          # dev sessions probed by the experiment cells (serve set / recall sweep)
BLIND_DATASET=f'{ORG}/TalkPlayData-Challenge-Blind-A'   # 80 sessions, NO gold

# ── retrieval ──
TOPK=500                   # fused-pool depth into the reranker
SUBMIT_K=20                # ids per submission row (official cap)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'

# ── ColBERT (the LoRA fine-tune from phase2_colbert_finetune) ──
# D_LEN / EXPANSION_FIRST MUST match phase2_colbert_finetune (the checkpoint was trained + indexed at these).
COLBERT_OUT_DIR=f'{OUT}/colbert/music-colbert-v1'   # merged plain-ColBERT checkpoint
Q_LEN=96; D_LEN=512; BSIZE=128; EXPANSION_FIRST=True
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_BASE='music-colbert-v1-d%d'%D_LEN

# ── K2 (LGBM) — LOAD ONLY. This notebook trains NOTHING; train K2 in phase2_rerank.ipynb. ──
# Point at the trained artifact (.txt + .features.json sidecar) if an experiment needs reranked scores.
# If that K2 used the frozen-CE feature, score the serve-side CE over the serve pool only (no train).
K2_MODEL_PATH=f'{OUT}/k2_lgbm.txt'
CE_MODEL='BAAI/bge-reranker-v2-m3'; CROSS_ENCODER_K=50; CE_MAX_DOC_TOKENS=480   # used only if K2 needs ce_score

import random as _r, numpy as _np
_r.seed(SEED); _np.random.seed(SEED)
try:
    import torch as _t; _t.manual_seed(SEED); _t.cuda.manual_seed_all(SEED)
except Exception: pass
print('MODE: DEV experiments (slot-free, trains nothing) | responder', RESPONDER, '| D_LEN', D_LEN, '| smoke', SMOKE)

## 4. Catalog + base channels + dense doc_mat

In [ ]:
import glob, os, pickle, hashlib, pandas as pd, numpy as np
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB)); assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names); CKNN_MODS={l:m for l,m in CONTENT_MODALITIES.items() if m in _avail}
te={l:TrackEmbeddings(tre.select_columns(['track_id',m]),modalities=[m]) for l,m in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')

model=SentenceTransformer(DENSE_MODEL,device='cuda')
# config-hashed cache key (model + enriched version + catalog size) — never silently load a stale matrix
_dm_sig=hashlib.md5(f'{DENSE_MODEL}|enriched={USE_ENRICHED}|{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}'.encode()).hexdigest()[:8]
DOC_MAT_NPY=f'{OUT}/dense_doc_mat_{_dm_sig}.npy'
if os.path.exists(DOC_MAT_NPY):
    doc_mat=np.load(DOC_MAT_NPY); print('loaded cached doc_mat', doc_mat.shape)
else:
    doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
    np.save(DOC_MAT_NPY, doc_mat); print('cached doc_mat ->', DOC_MAT_NPY)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
cooc=pickle.load(open(COOC_PKL,'rb')) if os.path.exists(COOC_PKL) else build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat))
if not os.path.exists(COOC_PKL): pickle.dump(cooc,open(COOC_PKL,'wb'))
cknn=[ContentKNNChannel(te[l],m,label=l) for l,m in CKNN_MODS.items()]
base_chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
            CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
qb_full=QueryBuilder()                    # base channels' serve query (full history)
qb_focused=QueryBuilder(recency_window=1) # ColBERT's focused query (train==serve)
print('base channels:', [c.label for c in base_chans])

## 5. ColBERT PLAID channel (fine-tuned, focused-query routed)

In [ ]:
from pylate import models
from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
from mcrs.retrieval.colbert_channel import colbert_doc_text
import hashlib

assert os.path.isdir(COLBERT_OUT_DIR), f'ColBERT checkpoint missing at {COLBERT_OUT_DIR} — run phase2_colbert_finetune first'
ft=models.ColBERT(model_name_or_path=COLBERT_OUT_DIR, query_length=Q_LEN, document_length=D_LEN)
doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=EXPANSION_FIRST)
# checkpoint-keyed PLAID index name so a retrain can't reuse a stale index (mismatched embedding spaces)
_ck_sig=hashlib.md5('|'.join(f'{f}:{os.stat(os.path.join(rt,f)).st_size}:{int(os.stat(os.path.join(rt,f)).st_mtime)}'
                             for rt,_,fs in os.walk(COLBERT_OUT_DIR) for f in sorted(fs)).encode()).hexdigest()[:8]
# doc-side sig: enriched catalog + D_LEN + EXPANSION_FIRST determine the doc vectors (same format as the
# fine-tune gate + blindA so the index is shared; a new parquet / D_LEN / expansion order busts it).
_doc_sig=hashlib.md5(f'{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}|d{D_LEN}|ef{EXPANSION_FIRST}'.encode()).hexdigest()[:8]
retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, f'{FT_IDX_BASE}-{_ck_sig}-{_doc_sig}', doc_fn, batch_size=BSIZE)

class PlaidColBERTChannel:
    """Fusion channel backed by the fine-tuned ColBERT PLAID index. query_key='colbert' so the harness /
    build_rerank_groups route it the focused query while base channels keep the full query."""
    label='colbert'; query_key='colbert'
    def __init__(self, model, retriever): self.model, self.retriever = model, retriever
    def batch_text_to_item_retrieval(self, queries, topk, batch_context=None, user_ids=None):
        if not queries: return []
        return colbert_retrieve(self.model, self.retriever, list(queries), topk, batch_size=BSIZE)

colbert=PlaidColBERTChannel(ft, retr)
chans=base_chans+[colbert]
fusion=RRFFusion(chans, k=60)
labels=[c.label for c in chans]
PCQ={'colbert': qb_focused}               # per-channel routing: ColBERT gets the focused query
print('fusion channels:', labels)

## Experiments
Each cell below is one slot-free experiment on the dev final-turn proxy.

In [ ]:
# === EXPERIMENT 1 — RECALL SWEEP (slot-free): warm-tuned weights + ColBERT promotion on dev FINAL turns ===
# No K2, no retrain — a few minutes. Measures the recall ceiling the way Blind-A is scored: the trailing
# turn per dev session (warm-heavy). Run the setup cells (catalog + channels + ColBERT) above first.
from mcrs.data.conversations import Conversations
from mcrs.eval.probe import recall_ceiling
from mcrs.eval.weight_sweep import segment_weight_sweep

SWEEP_N = min(DEV_SESSIONS, len(dsd['test']))   # dev sessions to probe
DEPTH   = 1000                                  # per-channel retrieval depth
KS      = [50, 200, 500]

conv_dev = Conversations(dsd['test'].select(range(SWEEP_N)))
dturns = list(conv_dev.gold_target_turns())  # dev final-turn proxy: trailing turn/session, has gold
golds    = [conv_dev.gold(t.session_id, t.turn_number) for t in dturns]
segments = [t.segment for t in dturns]
print(f'dev final-turn probe: {len(dturns)} turns | warm {segments.count("warm")} / cold {segments.count("cold")}')

q_full = [qb_full.build(t).text for t in dturns]
q_foc  = [qb_focused.build(t).text for t in dturns]
bc     = [{'history_tids': t.history_tids, 'user_id': t.user_id} for t in dturns]
uids   = [t.user_id for t in dturns]
per_channel = {}
for ch in chans:
    q = q_foc if getattr(ch, 'query_key', None) == 'colbert' else q_full   # ColBERT focused, rest full
    per_channel[ch.label] = ch.batch_text_to_item_retrieval(q, DEPTH, batch_context=bc, user_ids=uids)
    print('retrieved', ch.label)

def _w(d, k):  # safe nested get
    return d.get(k, float('nan')) if d else float('nan')
rep = recall_ceiling(per_channel, golds, KS, segments=segments)
print('\n--- per-channel: recall@500 | unique_recall | WARM recall@500 (the keep/drop signal) ---')
for lab in labels:
    e = rep['per_channel'][lab]
    print(f'  {lab:16s} r@500={e["recall"][500]:.3f}  unique={e["unique_recall"]:.4f}  warm@500={_w(e.get("by_segment",{}).get("warm",{}),500):.3f}')
fw = rep['fused']
print(f'  FUSED(uniform)   r@500={fw["recall"][500]:.3f}  warm@500={_w(fw.get("by_segment",{}).get("warm",{}),500):.3f}  warm@200={_w(fw.get("by_segment",{}).get("warm",{}),200):.3f}')

# candidate weight vectors: both levers at once — promote colbert / drop dense / up-weight personalization
def wv(**over): return {l: float(over.get(l, 1.0)) for l in labels}
pers = [l for l in labels if ('cf' in l or 'artist' in l)]   # cf-bpr, same_artist, related_artist
cands = [
    ('uniform',                    wv()),
    ('colbert_2x',                 wv(colbert=2.0)),
    ('drop_dense',                 wv(dense=0.0)),
    ('colbert_primary_drop_dense', wv(colbert=2.0, dense=0.0)),
    ('personalization_up',         wv(**{l: 1.5 for l in pers})),
    ('colbert_up+pers_up',         wv(colbert=2.0, **{l: 1.5 for l in pers})),
]
sweep = segment_weight_sweep(per_channel, golds, segments, cands, ks=(200, 500))
print('\n--- WARM segment sweep (Blind-A regime), ranked by recall@500 ---')
for name, rec in sweep.get('warm', []):
    print(f'  {name:28s} r@200={rec[200]:.3f}  r@500={rec[500]:.3f}')
print(f'\npers channels detected: {pers}')
print('READ: a config beating uniform warm r@500 by >~0.005 = real recall headroom -> worth a K2 confirm.')
print('      nothing beats uniform = the recall wall holds; weight/ColBERT-promotion is not the lever.')